# Spike de exploracion - `client_scaffold`

> Artefacto del **paso 4** (`notebook_writer`). Es un **spike**: explora y **visualiza** cada historia de usuario de `definition.md` con **datos sinteticos** para el gate humano del paso 5. **No es el producto**: al pasar a `app/src/` la spec + TDD reescriben la logica; este notebook queda como documentacion de referencia.

## Regla inviolable (C-01, Datos en Boveda)
Este notebook usa **exclusivamente datos sinteticos**. `clients_root` es un **directorio temporal** creado en tiempo de ejecucion; **jamas** se lee ni referencia `clients/<CLIENTE>/data/` real. Los nombres de tenant (`COMPANY_DEMO`, `Usuario 1`, `test1@correo.com`) son ficticios.

## Feature bajo prueba
- **Core:** `create_client(name, clients_root) -> Path` (el motor; la CLI es solo fachada).
- **CLI:** `zlk client new <NOMBRE_CLIENTE>` (se simula llamando al core).
- **Fuente canonica:** `system_design.md` §11-§12 + `feature_contract.md`.

## Mapa de trazabilidad (celda -> HU)
| Celda | Que demuestra | Traza |
|---|---|---|
| 1 | Setup: `clients_root` sintetico (temp dir) | - |
| 2 | Prototipo del core `create_client` + validacion de nombre | HU-02, HU-03 |
| 3 | Alta feliz: estructura canonica del tenant `COMPANY_DEMO` creada | HU-01, HU-06, HU-08 |
| 4 | Contenido de los placeholders (`client.yaml` + 3 YAMLs de `input/`) | HU-05 |
| 5 | `data/` medallion + `manifest.json` inicial vacio y valido | HU-06 |
| 6 | Simulacion de la CLI `zlk client new` (fachada delega al core) | HU-01, HU-02 |
| 7 | Nombre invalido -> error claro, sin artefactos parciales | HU-03 |
| 8 | Tenant existente -> falla (no idempotente), no modifica nada | HU-04 |
| 9 | C-01: `data/` cubierta por `.gitignore clients/*/data/` | HU-07 |
| 10 | `COMPANY_DEMO` vs otro tenant: misma estructura canonica (sin hardcode) | HU-08 |

## Celda 1 - Setup: `clients_root` sintetico
Traza: **-** (preparacion). Creamos un directorio temporal aislado que hace de `clients/` sintetico para no ensuciar el repo. Ningun dato real interviene.

In [1]:
import json
import re
import shutil
import tempfile
from datetime import datetime, timezone
from pathlib import Path

# clients_root SINTETICO: un temp dir aislado. Nunca tocamos clients/*/data/ real (C-01).
CLIENTS_ROOT = Path(tempfile.mkdtemp(prefix="zlk_spike_clients_"))
print(f"clients_root sintetico -> {CLIENTS_ROOT}")
print(f"existe y esta vacio: {CLIENTS_ROOT.is_dir()} / {list(CLIENTS_ROOT.iterdir())}")


def tree(path: Path, prefix: str = "") -> str:
    """Render simple de arbol de directorios para visualizar la estructura creada."""
    lines = []
    entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    for i, entry in enumerate(entries):
        connector = "|__ " if i == len(entries) - 1 else "|-- "
        suffix = "/" if entry.is_dir() else ""
        lines.append(f"{prefix}{connector}{entry.name}{suffix}")
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "|   "
            child = tree(entry, prefix + extension)
            if child:
                lines.append(child)
    return "\n".join(lines)

clients_root sintetico -> C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_clients_ivfshsg2
existe y esta vacio: True / []


## Celda 2 - Prototipo del core `create_client` + validacion estricta del nombre
Traza: **HU-02** (la logica vive en una funcion core reutilizable/testeable, sin CLI) y **HU-03** (validacion estricta del nombre).

### Regla de validacion (alineada con la spec aprobada)
Regla estricta alineada con `spec.md` (permite acentos y ñ del espanol):
- No vacio ni solo espacios.
- Alfabeto permitido: letras ASCII y acentuadas del espanol (á é í ó ú ü ñ y mayusculas), digitos, `_` y `-`. Sin espacios ni separadores de ruta (`/`, `\`, `.`), sin scripts no latinos.
- Longitud 1..64.
- Sin colision con un tenant existente (no idempotente, HU-04).

> Se propone una excepcion de dominio `ClientNameError` / `TenantExistsError` para que la CLI la traduzca a un mensaje claro y a un exit code != 0.

In [2]:
# --- Excepciones de dominio (propuesta para la spec) ---
class ClientNameError(ValueError):
    """El nombre del cliente no cumple la regla de validacion estricta."""


class TenantExistsError(FileExistsError):
    """Ya existe un tenant con ese nombre (operacion no idempotente)."""


# Regla de validacion (aprobada en el gate; alineada con spec.md)
NAME_PATTERN = re.compile(r"^[A-Za-z0-9_\-áéíóúüÁÉÍÓÚÜñÑ]{1,64}$")


def validate_client_name(name: str) -> str:
    """Valida el nombre del cliente. Lanza ClientNameError si es invalido."""
    if name is None or not str(name).strip():
        raise ClientNameError("El nombre del cliente no puede estar vacio.")
    if not NAME_PATTERN.match(name):
        raise ClientNameError(
            f"Nombre invalido: '{name}'. Permitido: letras ASCII y acentuadas del espanol "
            f"(incluida n con virgulilla), numeros, '_' y '-', longitud 1-64; "
            f"sin espacios ni separadores de ruta."
        )
    return name


# --- Plantillas de placeholders versionables (contenido sintetico, sin datos reales) ---
def _client_yaml(name: str) -> str:
    return (
        "# client.yaml - identidad del tenant (placeholder versionable)\n"
        "# Generado por create_client. Completar manualmente los campos pendientes.\n"
        f"client_id: {name}\n"
        f"display_name: \"{name}\"   # TODO: nombre visible del cliente\n"
        "sector_id: TODO_SECTOR   # TODO: id de sector (ver Maestro de Sectores, YAML 4)\n"
        "created_at: \"\"          # se completa al materializar\n"
    )


INPUT_PLACEHOLDERS = {
    "contract_data.yaml": (
        "# YAML 1 - Contrato de Datos (placeholder). Valida estructura del archivo.\n"
        "# Define columnas, tipos, nulos permitidos y llaves. Completar por archivo.\n"
        "version: 1\n"
        "archivos: []   # TODO: definir contratos por archivo del cliente\n"
    ),
    "business_rules.yaml": (
        "# YAML 2 - Reglas de Negocio (placeholder). Validaciones que cuestan dinero.\n"
        "version: 1\n"
        "reglas: []   # TODO: definir reglas (rangos, calculos derivados, integridad)\n"
    ),
    "finance.yaml": (
        "# YAML 3 - Variables Financieras (placeholder). Multiplicadores a USD.\n"
        "version: 1\n"
        "moneda: USD\n"
        "variables: {}   # TODO: salario_hora, ticket_promedio, %conversion, etc.\n"
        "facturacion_periodo: null   # opcional: habilita DHS Financiero (§8)\n"
    ),
}


def _empty_manifest() -> dict:
    """Ledger inicial vacio y valido (§11)."""
    return {"version": 1, "files": []}


def create_client(name: str, clients_root: Path) -> Path:
    """Materializa la estructura canonica de un tenant en disco (§11).

    - Valida el nombre (estricto).
    - Falla si el tenant ya existe (no idempotente, sin --force).
    - Es transaccional: ante error no deja artefactos parciales.
    - Retorna la ruta del tenant creado.
    """
    name = validate_client_name(name)
    clients_root = Path(clients_root)
    tenant = clients_root / name

    if tenant.exists():
        raise TenantExistsError(
            f"El tenant '{name}' ya existe en {clients_root}. "
            f"Operacion no idempotente: no se sobrescribe (no hay --force)."
        )

    # Construccion en carpeta temporal + rename atomico -> sin tenant a medias (HU-03).
    staging = clients_root / f".{name}.staging"
    if staging.exists():
        shutil.rmtree(staging)
    try:
        (staging / "input").mkdir(parents=True)
        for layer in ("bronze", "silver", "gold"):
            (staging / "data" / layer).mkdir(parents=True)

        (staging / "client.yaml").write_text(_client_yaml(name), encoding="utf-8")
        for fname, content in INPUT_PLACEHOLDERS.items():
            (staging / "input" / fname).write_text(content, encoding="utf-8")
        (staging / "data" / "manifest.json").write_text(
            json.dumps(_empty_manifest(), indent=2, ensure_ascii=False), encoding="utf-8"
        )
        staging.rename(tenant)  # commit atomico
    except Exception:
        if staging.exists():
            shutil.rmtree(staging)
        raise
    return tenant


print("Core prototipado: create_client, validate_client_name, excepciones de dominio.")
print("Prueba rapida de validacion (nombres validos):")
for ok in ["COMPANY_DEMO", "cliente_01", "Café_Málaga", "Sanducheria_Ñoño"]:
    print(f"  '{ok}' -> valido: {bool(NAME_PATTERN.match(ok))}")

Core prototipado: create_client, validate_client_name, excepciones de dominio.
Prueba rapida de validacion (nombres validos):
  'COMPANY_DEMO' -> valido: True
  'cliente_01' -> valido: True
  'Café_Málaga' -> valido: True
  'Sanducheria_Ñoño' -> valido: True


## Celda 3 - Alta feliz: estructura canonica del tenant
Traza: **HU-01** (crea el tenant completo y reporta la ruta), **HU-06** (nace con capas medallion) y **HU-08** (creamos `COMPANY_DEMO`, nuestro tenant de demostracion, por la via oficial). Visualizamos el arbol de directorios creado.

In [3]:
tenant_path = create_client("COMPANY_DEMO", CLIENTS_ROOT)
print(f"[OK] Tenant creado -> {tenant_path}\n")
print("Estructura generada:")
print(f"{tenant_path.name}/")
print(tree(tenant_path))

# Verificacion visible de la estructura canonica esperada (§11)
esperado = [
    "client.yaml",
    "input/contract_data.yaml",
    "input/business_rules.yaml",
    "input/finance.yaml",
    "data/bronze",
    "data/silver",
    "data/gold",
    "data/manifest.json",
]
print("\nChecklist de estructura canonica:")
for rel in esperado:
    print(f"  [{'x' if (tenant_path / rel).exists() else ' '}] {rel}")

[OK] Tenant creado -> C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_clients_ivfshsg2\COMPANY_DEMO

Estructura generada:
COMPANY_DEMO/
|-- data/
|   |-- bronze/
|   |-- gold/
|   |-- silver/
|   |__ manifest.json
|-- input/
|   |-- contract_data.yaml
|   |-- finance.yaml
|   |__ business_rules.yaml
|__ client.yaml

Checklist de estructura canonica:
  [x] client.yaml
  [x] input/contract_data.yaml
  [x] input/business_rules.yaml
  [x] input/finance.yaml
  [x] data/bronze
  [x] data/silver
  [x] data/gold
  [x] data/manifest.json


## Celda 4 - Contenido de los placeholders versionables
Traza: **HU-05**. Mostramos `client.yaml` (identidad + `sector_id`) y los 3 YAMLs de `input/` (contrato, reglas, finanzas), y validamos que su sintaxis YAML sea correcta.

In [4]:
archivos_config = [
    tenant_path / "client.yaml",
    tenant_path / "input" / "contract_data.yaml",
    tenant_path / "input" / "business_rules.yaml",
    tenant_path / "input" / "finance.yaml",
]

for f in archivos_config:
    print(f"===== {f.relative_to(tenant_path)} =====")
    print(f.read_text(encoding="utf-8"))

# Validacion de sintaxis YAML (HU-05 exige placeholders con sintaxis valida)
try:
    import yaml  # PyYAML
    print("Validacion de sintaxis YAML:")
    for f in archivos_config:
        yaml.safe_load(f.read_text(encoding="utf-8"))
        print(f"  [OK] {f.relative_to(tenant_path)} parsea correctamente")
except ImportError:
    print("[nota] PyYAML no instalado en este entorno: se omite el parseo. ")
    print("       La spec debera verificar sintaxis YAML valida en los placeholders.")

===== client.yaml =====
# client.yaml - identidad del tenant (placeholder versionable)
# Generado por create_client. Completar manualmente los campos pendientes.
client_id: COMPANY_DEMO
display_name: "COMPANY_DEMO"   # TODO: nombre visible del cliente
sector_id: TODO_SECTOR   # TODO: id de sector (ver Maestro de Sectores, YAML 4)
created_at: ""          # se completa al materializar

===== input\contract_data.yaml =====
# YAML 1 - Contrato de Datos (placeholder). Valida estructura del archivo.
# Define columnas, tipos, nulos permitidos y llaves. Completar por archivo.
version: 1
archivos: []   # TODO: definir contratos por archivo del cliente

===== input\business_rules.yaml =====
# YAML 2 - Reglas de Negocio (placeholder). Validaciones que cuestan dinero.
version: 1
reglas: []   # TODO: definir reglas (rangos, calculos derivados, integridad)

===== input\finance.yaml =====
# YAML 3 - Variables Financieras (placeholder). Multiplicadores a USD.
version: 1
moneda: USD
variables: {}   # T

## Celda 5 - `data/` medallion + `manifest.json` inicial
Traza: **HU-06**. Verificamos las 3 capas medallion vacias y que el `manifest.json` es un ledger inicial **vacio y valido**.

In [5]:
data_dir = tenant_path / "data"
print("Capas medallion (deben existir y estar vacias en el alta):")
for layer in ("bronze", "silver", "gold"):
    d = data_dir / layer
    print(f"  [{'x' if d.is_dir() else ' '}] data/{layer}/  -> contenido: {list(d.iterdir())}")

manifest_path = data_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("\ndata/manifest.json (ledger inicial):")
print(json.dumps(manifest, indent=2, ensure_ascii=False))
print(f"\nledger vacio y valido: {manifest.get('files') == []} (files == [])")

Capas medallion (deben existir y estar vacias en el alta):
  [x] data/bronze/  -> contenido: []
  [x] data/silver/  -> contenido: []
  [x] data/gold/  -> contenido: []

data/manifest.json (ledger inicial):
{
  "version": 1,
  "files": []
}

ledger vacio y valido: True (files == [])


## Celda 6 - Simulacion de la CLI `zlk client new`
Traza: **HU-01** y **HU-02**. La CLI es **solo una fachada**: parsea el argumento, delega toda la logica al core y traduce excepciones a un mensaje + exit code. Aqui simulamos ese comportamiento sin lanzar un proceso real.

In [6]:
def cli_client_new(name: str, clients_root: Path) -> int:
    """Fachada CLI simulada de 'zlk client new <NOMBRE>'. Devuelve exit code."""
    try:
        path = create_client(name, clients_root)  # toda la logica vive en el core
        print(f"[zlk] Cliente '{name}' creado en: {path}")
        return 0
    except ClientNameError as e:
        print(f"[zlk] ERROR (nombre invalido): {e}")
        return 2
    except TenantExistsError as e:
        print(f"[zlk] ERROR (ya existe): {e}")
        return 3


print("$ zlk client new CAFE_CENTRAL")
code = cli_client_new("CAFE_CENTRAL", CLIENTS_ROOT)
print(f"exit code = {code}")
print("\nMisma via, mismo resultado que el core directo -> la CLI no tiene logica propia (HU-02).")
print(f"tenant creado por CLI simulada existe: {(CLIENTS_ROOT / 'CAFE_CENTRAL').is_dir()}")

$ zlk client new CAFE_CENTRAL
[zlk] Cliente 'CAFE_CENTRAL' creado en: C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_clients_ivfshsg2\CAFE_CENTRAL
exit code = 0

Misma via, mismo resultado que el core directo -> la CLI no tiene logica propia (HU-02).
tenant creado por CLI simulada existe: True


## Celda 7 - Nombre invalido: error claro, sin artefactos parciales
Traza: **HU-03**. Probamos varios nombres invalidos: cada uno falla con mensaje claro y **no** deja rastro en `clients_root`.

In [7]:
invalidos = [
    ("", "vacio"),
    ("   ", "solo espacios"),
    ("cliente con espacios", "contiene espacios"),
    ("cli/ente", "separador de ruta"),
    ("../escape", "path traversal"),
    ("клиент", "script no latino (cirilico)"),
    ("cafe😀", "emoji fuera del alfabeto permitido"),
    ("x" * 65, "excede 64 chars"),
]

antes = set(p.name for p in CLIENTS_ROOT.iterdir())
for nombre, motivo in invalidos:
    try:
        create_client(nombre, CLIENTS_ROOT)
        print(f"  [FALLO DEL TEST] '{nombre[:20]}' ({motivo}) fue aceptado (no deberia)")
    except ClientNameError as e:
        print(f"  [OK] rechazado ({motivo}): {str(e)[:70]}...")

despues = set(p.name for p in CLIENTS_ROOT.iterdir())
residuo = despues - antes
print(f"\nArtefactos nuevos tras nombres invalidos: {residuo if residuo else 'NINGUNO (limpio)'}")
print("No quedaron tenants a medias ni carpetas .staging. (HU-03)")

  [OK] rechazado (vacio): El nombre del cliente no puede estar vacio....
  [OK] rechazado (solo espacios): El nombre del cliente no puede estar vacio....
  [OK] rechazado (contiene espacios): Nombre invalido: 'cliente con espacios'. Permitido: letras ASCII y ace...
  [OK] rechazado (separador de ruta): Nombre invalido: 'cli/ente'. Permitido: letras ASCII y acentuadas del ...
  [OK] rechazado (path traversal): Nombre invalido: '../escape'. Permitido: letras ASCII y acentuadas del...
  [OK] rechazado (script no latino (cirilico)): Nombre invalido: 'клиент'. Permitido: letras ASCII y acentuadas del es...
  [OK] rechazado (emoji fuera del alfabeto permitido): Nombre invalido: 'cafe😀'. Permitido: letras ASCII y acentuadas del esp...
  [OK] rechazado (excede 64 chars): Nombre invalido: 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...

Artefactos nuevos tras nombres invalidos: NINGUNO (limpio)
No quedaron tenants a medias ni carpetas .staging. (HU-03)


## Celda 8 - Tenant existente: falla no idempotente
Traza: **HU-04**. Reintentar sobre un tenant ya creado falla con error claro y **no** modifica el tenant existente.

In [8]:
# Firma del estado actual del tenant COMPANY_DEMO antes del reintento
def snapshot(path: Path) -> dict:
    return {
        str(p.relative_to(path)): (p.stat().st_size if p.is_file() else "<dir>")
        for p in sorted(path.rglob("*"))
    }


antes_snap = snapshot(tenant_path)
print("$ zlk client new COMPANY_DEMO   (segundo intento sobre tenant existente)")
code = cli_client_new("COMPANY_DEMO", CLIENTS_ROOT)
print(f"exit code = {code} (!= 0 esperado)\n")

despues_snap = snapshot(tenant_path)
print(f"El tenant existente quedo intacto: {antes_snap == despues_snap}")
print(f"  archivos/carpetas antes: {len(antes_snap)} | despues: {len(despues_snap)}")

$ zlk client new COMPANY_DEMO   (segundo intento sobre tenant existente)
[zlk] ERROR (ya existe): El tenant 'COMPANY_DEMO' ya existe en C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_clients_ivfshsg2. Operacion no idempotente: no se sobrescribe (no hay --force).
exit code = 3 (!= 0 esperado)

El tenant existente quedo intacto: True
  archivos/carpetas antes: 10 | despues: 10


## Celda 9 - C-01: `data/` cubierta por `.gitignore clients/*/data/`
Traza: **HU-07** (Datos en Boveda). Demostramos que la regla `.gitignore` `clients/*/data/` casa con la ruta `data/` del tenant y **no** con `client.yaml`/`input/`. Simulamos el matching del patron para no depender de git.

In [9]:
import fnmatch

# Patron real vigente en el .gitignore del repo (linea 34)
GITIGNORE_RULE = "clients/*/data/"


def ignora(rel_posix: str, rule: str = GITIGNORE_RULE) -> bool:
    """Aproxima el matching de gitignore para la regla 'clients/*/data/'.
    Un path se ignora si el prefijo de directorio 'clients/<X>/data' matchea."""
    prefix = rule.rstrip("/")  # clients/*/data
    return fnmatch.fnmatch(rel_posix, prefix) or fnmatch.fnmatch(rel_posix, prefix + "/*")


# Rutas relativas al repo tal como se verian para el tenant COMPANY_DEMO
rutas = [
    "clients/COMPANY_DEMO/client.yaml",
    "clients/COMPANY_DEMO/input/finance.yaml",
    "clients/COMPANY_DEMO/data",
    "clients/COMPANY_DEMO/data/bronze/ventas_pos.csv",
    "clients/COMPANY_DEMO/data/manifest.json",
    "clients/COMPANY_DEMO/data/gold/2026-Q3/pareto.csv",
]

print(f"Regla .gitignore evaluada: '{GITIGNORE_RULE}'\n")
print(f"{'IGNORADO':>9} | ruta")
print("-" * 60)
for r in rutas:
    print(f"{'SI' if ignora(r) else 'no':>9} | {r}")

print("\nConclusion (HU-07 / C-01):")
print("  - Todo lo bajo data/ (bronze/silver/gold/manifest) queda IGNORADO -> nunca entra a git.")
print("  - client.yaml e input/ SI se versionan (config sin PII).")

Regla .gitignore evaluada: 'clients/*/data/'

 IGNORADO | ruta
------------------------------------------------------------
       no | clients/COMPANY_DEMO/client.yaml
       no | clients/COMPANY_DEMO/input/finance.yaml
       SI | clients/COMPANY_DEMO/data
       SI | clients/COMPANY_DEMO/data/bronze/ventas_pos.csv
       SI | clients/COMPANY_DEMO/data/manifest.json
       SI | clients/COMPANY_DEMO/data/gold/2026-Q3/pareto.csv

Conclusion (HU-07 / C-01):
  - Todo lo bajo data/ (bronze/silver/gold/manifest) queda IGNORADO -> nunca entra a git.
  - client.yaml e input/ SI se versionan (config sin PII).


## Celda 10 - Tenant de demostracion `COMPANY_DEMO` + generalidad multi-tenant
Traza: **HU-08**. `COMPANY_DEMO` se creo en la Celda 3 por la **misma via oficial** (`create_client`): es nuestro tenant de demostracion. Aqui confirmamos que su estructura canonica es **identica** a la de cualquier otro tenant (`CAFE_CENTRAL`), demostrando que la via es unica y sin hardcode (multi-tenant, §12).

In [10]:
# COMPANY_DEMO ya se creo en la Celda 3 por la via oficial `create_client` (HU-08):
# es nuestro tenant de demostracion. Aqui verificamos la GENERALIDAD multi-tenant (§12):
# cualquier otro tenant creado por la misma via obtiene una estructura canonica identica,
# de modo que nada esta hardcodeado a un cliente concreto.
demo_path = tenant_path                      # COMPANY_DEMO, nuestro tenant de demostracion
otro_path = CLIENTS_ROOT / "CAFE_CENTRAL"    # segundo tenant, creado via CLI en la Celda 6


def estructura_rel(path: Path) -> set:
    return {str(p.relative_to(path)).replace("\\", "/") for p in path.rglob("*")}


print(f"[OK] Tenant de demostracion -> {demo_path}")
print(f"{demo_path.name}/")
print(tree(demo_path))
print()

iguales = estructura_rel(demo_path) == estructura_rel(otro_path)
print(f"COMPANY_DEMO y CAFE_CENTRAL tienen estructura canonica identica: {iguales}")
print()
print("Tenants presentes en el clients_root sintetico:")
for t in sorted(p.name for p in CLIENTS_ROOT.iterdir() if p.is_dir()):
    print(f"  - {t}")

[OK] Tenant de demostracion -> C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_clients_ivfshsg2\COMPANY_DEMO
COMPANY_DEMO/
|-- data/
|   |-- bronze/
|   |-- gold/
|   |-- silver/
|   |__ manifest.json
|-- input/
|   |-- contract_data.yaml
|   |-- finance.yaml
|   |__ business_rules.yaml
|__ client.yaml

COMPANY_DEMO y CAFE_CENTRAL tienen estructura canonica identica: True

Tenants presentes en el clients_root sintetico:
  - CAFE_CENTRAL
  - COMPANY_DEMO


## Celda 11 - Limpieza del entorno sintetico
Traza: **-**. Eliminamos el directorio temporal. Nada de esto toca el repo ni datos reales.

In [11]:
shutil.rmtree(CLIENTS_ROOT, ignore_errors=True)
print(f"clients_root sintetico eliminado: {not CLIENTS_ROOT.exists()}")
print("Spike completado. Ningun dato real intervino en ningun momento (C-01).")

clients_root sintetico eliminado: True
Spike completado. Ningun dato real intervino en ningun momento (C-01).


## Resumen del spike (para la spec, paso 6)

**Cobertura de historias** (todas con resultado visible):
- HU-01 (celdas 3, 6): alta crea estructura completa y reporta ruta.
- HU-02 (celdas 2, 6): logica en core `create_client`; la CLI solo delega.
- HU-03 (celdas 2, 7): validacion estricta; error claro; sin artefactos parciales (staging + rename atomico).
- HU-04 (celda 8): no idempotente; tenant existente intacto.
- HU-05 (celda 4): `client.yaml` + 3 YAMLs de `input/` como placeholders con sintaxis valida.
- HU-06 (celdas 3, 5): capas medallion + `manifest.json` ledger vacio valido.
- HU-07 (celda 9): `data/` cubierta por `.gitignore clients/*/data/`.
- HU-08 (celda 10): `COMPANY_DEMO` por la misma via, estructura canonica identica.

**Hallazgos / decisiones para la spec:**
1. **Regla de nombre (aprobada en el gate):** `^[A-Za-z0-9_\-áéíóúüÁÉÍÓÚÜñÑ]{1,64}$`. Permite letras ASCII y acentuadas del espanol + ñ, digitos, `_` y `-`; bloquea vacios, espacios, separadores de ruta, path traversal y scripts no latinos (cirilico, emoji).
2. **Atomicidad:** construir en carpeta `.staging` y hacer `rename` al final garantiza que un fallo no deje tenant a medias (HU-03). La spec debe fijar este contrato de "todo o nada".
3. **Excepciones de dominio** (`ClientNameError`, `TenantExistsError`) mapeadas por la CLI a exit codes distintos (2 y 3) y mensajes claros.
4. **manifest.json** minimo: `{"version": 1, "files": []}`. La spec puede enriquecer el schema del ledger.
5. **Placeholders YAML:** contenido comentado con TODOs; PyYAML valida sintaxis. Confirmar en spec el contenido minimo exacto de cada YAML.
6. **Riesgo tecnico:** `rename` atomico depende de estar en el mismo filesystem; el core produccion debe garantizar que staging vive dentro de `clients_root`.

> **Siguiente paso: GATE HUMANO (paso 5).** El humano revisa y aprueba este spike antes de pasar a `spec_writer` (paso 6). Este notebook es referencia; la logica se reescribe con rigor via spec + TDD (no se copia-pega).